[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/math_module2/blob/main/notebooks/ch3_derivative.ipynb)

# Chapter 3 — Companion Notebook
## The Slope of a Single Point
*How fast is something changing *right now*, when "rate" seems to need two moments?*

Three live pictures of the central act of calculus: a **secant that becomes a tangent** as the step $h$ shrinks to nothing, the **derivative traced out as its own function** while a cursor sweeps the curve, and a **corner/cusp zoom** showing the one place a slope can fail to exist.

> **How to use this notebook.** Run every cell from the top (Shift+Enter). Each widget below is live — drag the sliders and watch the picture respond. Requires `ipywidgets` and `matplotlib`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from ipywidgets import (interact, interactive, fixed, Dropdown, Checkbox,
                        Button, Output, VBox, HBox,
                        FloatSlider, IntSlider, FloatLogSlider)
from IPython.display import HTML, display, Markdown

# Book 2 palette — matches the printed figures in figures/png/
BLUE="#1f4e79"; RED="#c0392b"; GREEN="#2e8b57"; ORANGE="#e08e0b"
PURPLE="#6c3483"; TEAL="#1b9e9e"; GRAY="#7f8c8d"; DARK="#2c3e50"
SHADE="#9fc5e8"; SHADE2="#f4b6ad"; SHADEG="#a9dfbf"; LIGHT="#d6dbdf"

plt.rcParams.update({
    "figure.figsize": (7.2, 4.5), "figure.dpi": 96,
    "font.family": "serif", "mathtext.fontset": "cm", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": LIGHT, "grid.linewidth": 0.7,
    "lines.linewidth": 2.2, "lines.solid_capstyle": "round",
})

def axes(ax=None, title=None):
    '''A clean axes with a light grid; returns the axes.'''
    if ax is None:
        _, ax = plt.subplots()
    if title:
        ax.set_title(title, color=DARK)
    return ax


## 1. The secant becomes the tangent

Here is the move that creates calculus. The slope between two points $(a,f(a))$ and $(a+h,\,f(a+h))$ is the **secant slope** — the average rate of change over the interval:

$$\frac{f(a+h)-f(a)}{h}.$$

Now slide the second point in: shrink $h$ toward $0$. The two points merge, the gray secant **pivots**, and it settles onto the red **tangent** line — the slope at the single instant $a$. Watch the live secant slope chase the true derivative $f'(a)$ as you drag $h$ small. (Try $h$ both positive and negative — the secant swings in from either side and lands on the same line.)

In [ ]:
# Each preset carries f, its hand-computed derivative f', a label,
# and a window. Derivatives are exactly the ones earned in the
# chapter: (x^2)'=2x, (sin x)'=cos x, (sqrt x)'=1/(2 sqrt x).
_F1 = {
    'x^2':
        dict(f=lambda x: x**2, df=lambda x: 2*x,
             tex='x^2', dtex='2a', xlim=(-3.0, 3.0)),
    'sin(x)':
        dict(f=np.sin, df=np.cos,
             tex=r'\sin x', dtex=r'\cos a', xlim=(-1.0, 6.5)),
    'sqrt(x)':
        dict(f=np.sqrt, df=lambda x: 0.5/np.sqrt(x),
             tex=r'\sqrt{x}', dtex=r'1/(2\sqrt{a})',
             xlim=(0.05, 6.0)),
}

def _secant(which='x^2', a=1.0, h=1.5):
    p = _F1[which]
    f, df = p['f'], p['df']
    lo, hi = p['xlim']
    # Keep a, and a+h, inside the domain window.
    a = float(np.clip(a, lo + 0.05, hi - 0.05))
    if a + h < lo + 0.02:
        h = (lo + 0.02) - a
    if a + h > hi - 0.02:
        h = (hi - 0.02) - a

    xs = np.linspace(lo, hi, 600)
    fa, fah = f(a), f(a + h)
    slope_true = df(a)

    fig, ax = plt.subplots()
    ax.plot(xs, f(xs), color=BLUE, lw=2.4, label=rf'$f(x)={p["tex"]}$')
    # True tangent at a (red).
    ax.plot(xs, fa + slope_true*(xs - a), color=RED, lw=2.0,
            label='tangent (slope $f\'(a)$)')
    # Secant through the two points (gray) — only if h != 0.
    if abs(h) > 1e-9:
        slope_sec = (fah - fa)/h
        ax.plot(xs, fa + slope_sec*(xs - a), color=GRAY, lw=2.0,
                ls='--', label='secant')
        ax.plot([a + h], [fah], 'o', color=GRAY, ms=8, zorder=5)
    else:
        slope_sec = slope_true
    ax.plot([a], [fa], 'o', color=RED, ms=8, zorder=6)

    ax.set_xlim(lo, hi)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    ax.set_title('Secant pivoting onto the tangent', color=DARK)
    ax.legend(loc='upper left', framealpha=0.9)
    plt.show()

    gap = slope_sec - slope_true
    display(Markdown(
        rf'secant slope $=\dfrac{{f(a+h)-f(a)}}{{h}}'
        rf'= {slope_sec:.4f}$ &nbsp;&nbsp; vs &nbsp;&nbsp; '
        rf'true $f\'(a)={p["dtex"]}={slope_true:.4f}$ '
        rf'&nbsp;&nbsp;(gap ${gap:+.4f}$)'))

interact(_secant,
         which=Dropdown(options=list(_F1), value='x^2',
                        description='function'),
         a=FloatSlider(value=1.0, min=-2.5, max=2.5, step=0.1,
                       description='base $a$'),
         h=FloatSlider(value=1.5, min=-2.0, max=2.0, step=0.05,
                       description='step $h$'));


interactive(children=(Dropdown(description='function', options=('x^2', 'sin(x)', 'sqrt(x)'), value='x^2'), Flo…

## 2. The derivative is itself a function

The slope $f'(a)$ isn't a single number stuck to one point — it's a **rule** that returns the slope at *every* point. Watch that rule get built. Slide the cursor $x_0$ across the curve; the top panel shows $f$ with its red tangent at $x_0$, and the bottom panel plots the slope-function $f'(x)$ with a moving dot at $(x_0,\,f'(x_0))$.

As the cursor moves, the dot traces the lower curve point by point: where $f$ is steep, $f'$ is large; where $f$ is flat, $f'$ crosses zero. The bottom graph *is* the slope of the top one.

In [3]:
# f, f', and a window. f(x) = (1/3)x^3 - x has slope f'(x)=x^2-1,
# a clean parabola that crosses zero at the two turning points.
_F2 = {
    '(1/3)x^3 - x':
        dict(f=lambda x: x**3/3 - x, df=lambda x: x**2 - 1,
             tex=r'\frac{1}{3}x^3 - x', dtex='x^2-1',
             xlim=(-2.4, 2.4)),
    'sin(x)':
        dict(f=np.sin, df=np.cos,
             tex=r'\sin x', dtex=r'\cos x', xlim=(-0.5, 6.8)),
    'x^2':
        dict(f=lambda x: x**2, df=lambda x: 2*x,
             tex='x^2', dtex='2x', xlim=(-3.0, 3.0)),
}

def _slopefn(which='(1/3)x^3 - x', x0=0.6):
    p = _F2[which]
    f, df = p['f'], p['df']
    lo, hi = p['xlim']
    x0 = float(np.clip(x0, lo, hi))
    xs = np.linspace(lo, hi, 600)
    m = df(x0)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 6.4),
                                   sharex=True)
    # Top: f with the tangent at x0.
    ax1.plot(xs, f(xs), color=BLUE, lw=2.4, label=rf'$f={p["tex"]}$')
    tx = np.linspace(x0 - 0.9, x0 + 0.9, 2)
    ax1.plot(tx, f(x0) + m*(tx - x0), color=RED, lw=2.2,
             label='tangent at $x_0$')
    ax1.plot([x0], [f(x0)], 'o', color=RED, ms=8, zorder=5)
    ax1.axvline(x0, color=GRAY, lw=0.8, ls=':')
    ax1.set_ylabel('$f(x)$')
    ax1.set_title('Top: the function and its tangent', color=DARK)
    ax1.legend(loc='upper left', framealpha=0.9)
    # Bottom: f' with the moving dot.
    ax2.plot(xs, df(xs), color=RED, lw=2.4,
             label=rf"$f'(x)={p['dtex']}$")
    ax2.axhline(0, color=GRAY, lw=0.8)
    ax2.axvline(x0, color=GRAY, lw=0.8, ls=':')
    ax2.plot([x0], [m], 'o', color=RED, ms=9, zorder=5)
    ax2.annotate(rf'$f\'(x_0)={m:.2f}$', xy=(x0, m),
                 xytext=(8, 8), textcoords='offset points',
                 color=RED, fontsize=11)
    ax2.set_xlabel('$x$'); ax2.set_ylabel("$f'(x)$")
    ax2.set_title('Bottom: the slope traced as a function',
                  color=DARK)
    ax2.legend(loc='upper left', framealpha=0.9)
    ax2.set_xlim(lo, hi)
    plt.tight_layout()
    plt.show()

interact(_slopefn,
         which=Dropdown(options=list(_F2), value='(1/3)x^3 - x',
                        description='function'),
         x0=FloatSlider(value=0.6, min=-2.3, max=2.3, step=0.05,
                        description='cursor $x_0$'));


interactive(children=(Dropdown(description='function', options=('(1/3)x^3 - x', 'sin(x)', 'x^2'), value='(1/3)…

## 3. Where the slope fails: corners and cusps

A derivative is a *limit*, and limits don't always exist — so neither do derivatives. Zoom in on $x=0$ for two continuous functions that both refuse a slope there, but for different reasons.

- **$|x|$** keeps a sharp **corner** at every zoom — it never straightens out. Its difference quotient $\dfrac{|h|}{h}$ is $+1$ from the right and $-1$ from the left: the two one-sided limits disagree, so $f'(0)$ does not exist. (This is exactly the $\tfrac{|x|}{x}$ jump from Chapter 2, reappearing as a slope.)
- **$x^{1/3}$** is smooth but develops a **vertical tangent**: as you zoom, it stands straight up, so the slope runs off to $+\infty$.

Drag the **zoom** slider (a log scale): the smaller the window, the closer you look at the origin.

In [4]:
def _cbrt(x):
    # Real cube root, valid for negative x (np.cbrt is real-valued).
    return np.cbrt(x)

_F3 = {
    '|x|  (corner)':
        dict(f=np.abs, tex='|x|', kind='corner'),
    'x^(1/3)  (cusp / vertical tangent)':
        dict(f=_cbrt, tex='x^{1/3}', kind='cusp'),
}

def _corner(which='|x|  (corner)', zoom=1.0, show_quotient=True):
    p = _F3[which]
    f = p['f']
    w = float(zoom)                 # half-width of the window
    xs = np.linspace(-w, w, 601)
    fig, ax = plt.subplots()
    ax.plot(xs, f(xs), color=BLUE, lw=2.6, label=rf'$f(x)={p["tex"]}$')
    ax.axvline(0, color=GRAY, lw=0.8)
    ax.axhline(0, color=GRAY, lw=0.8)
    ax.plot([0], [f(0.0)], 'o', color=RED, ms=8, zorder=5)
    ax.set_xlim(-w, w)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
    note = ('corner stays sharp' if p['kind'] == 'corner'
            else 'curve stands vertical')
    ax.set_title(f'zoom half-width = {w:.3g}   ({note})',
                 color=DARK)
    ax.legend(loc='upper left', framealpha=0.9)
    plt.show()

    if p['kind'] == 'corner' and show_quotient:
        hpos, hneg = w/2, -w/2
        qpos = (abs(hpos) - 0.0)/hpos
        qneg = (abs(hneg) - 0.0)/hneg
        display(Markdown(
            rf'difference quotient $\dfrac{{|h|}}{{h}}$ at $a=0$: '
            rf'right ($h>0$) $={qpos:+.0f}$, &nbsp; '
            rf'left ($h<0$) $={qneg:+.0f}$ — they disagree, so '
            rf'$f\'(0)$ does **not** exist.'))
    elif p['kind'] == 'cusp':
        hsmall = w/2
        slope = (_cbrt(hsmall) - 0.0)/hsmall
        display(Markdown(
            rf'secant slope across $[0,\,{hsmall:.3g}]$ is '
            rf'$\approx {slope:.2f}$ — it grows without bound as '
            rf'you zoom in: a **vertical tangent**, slope $\to'
            rf'+\infty$.'))

interact(_corner,
         which=Dropdown(options=list(_F3), value='|x|  (corner)',
                        description='function'),
         zoom=FloatLogSlider(value=1.0, base=10, min=-3, max=0.3,
                             step=0.1, description='zoom'),
         show_quotient=Checkbox(value=True,
                                description='show |h|/h'));


interactive(children=(Dropdown(description='function', options=('|x|  (corner)', 'x^(1/3)  (cusp / vertical ta…

---

*One limit — the slope of a single point — and the rest of the book is its consequences. Next, **Chapter 4** proves a handful of rules so you never have to expand $(a+h)^n$ by hand again; and the one-sided quotients you just watched fail at the corner of $|x|$ return as Fermat's lemma in Chapter 6.*